# Steam Oyun Verisi Temizleme

Bu notebookta veri inceleme aşamasında belirlenen kurallar uygulanarak
öneri sisteminde kullanılacak temiz oyun veri seti oluşturulacaktır.

Ham veri dosyası değiştirilmeyecek, temizlenmiş veri ayrı bir dosya olarak
`data/processed/` klasörüne kaydedilecektir.


In [1]:
import pandas as pd

dosya_yolu = "../data/raw/games.csv"

sutunlar = pd.read_csv(
    dosya_yolu,
    nrows=0
).columns.tolist()

birlesik_sutun = sutunlar.index("DiscountDLC count")

sutunlar[
    birlesik_sutun:birlesik_sutun + 1
] = [
    "Discount",
    "DLC count"
]

veri = pd.read_csv(
    dosya_yolu,
    skiprows=1,
    names=sutunlar
)

print("Başlangıç kayıt sayısı:", len(veri))
print("Sütun sayısı:", len(veri.columns))

Başlangıç kayıt sayısı: 125855
Sütun sayısı: 40


## Playtest kayıtlarının çıkarılması

Playtest kayıtları bağımsız ve kalıcı oyunlar olmayabildiği için temizlenmiş
öneri havuzuna alınmayacaktır. Ham veri korunacak, yalnızca temiz veri
oluşturulurken bu kayıtlar dışarıda bırakılacaktır.

In [2]:
playtest_mi = veri["Name"].fillna("").str.contains(
    "Playtest",
    case=False,
    na=False
)

playtest_kayitlari = veri[
    playtest_mi
]

temiz_veri = veri[
    playtest_mi == False
].copy()

print("Başlangıç kayıt sayısı:", len(veri))
print("Çıkarılan Playtest sayısı:", len(playtest_kayitlari))
print("Kalan kayıt sayısı:", len(temiz_veri))

Başlangıç kayıt sayısı: 125855
Çıkarılan Playtest sayısı: 8139
Kalan kayıt sayısı: 117716


## Temel alanları eksik kayıtların çıkarılması

Oyun adı, açıklama, kategori ve tür alanları öneri sisteminin temel
bilgileridir. Bu alanlardan en az biri eksik olan kayıtlar temizlenmiş
öneri havuzuna alınmayacaktır.

In [3]:
temel_alanlar = [
    "Name",
    "About the game",
    "Categories",
    "Genres"
]

eksik_temel_mi = temiz_veri[
    temel_alanlar
].isna().any(axis=1)

eksik_temel_kayitlar = temiz_veri[
    eksik_temel_mi
]

temiz_veri = temiz_veri[
    eksik_temel_mi == False
].copy()

print("Playtest sonrası kayıt sayısı:", len(eksik_temel_kayitlar) + len(temiz_veri))
print("Çıkarılan eksik alanlı kayıt:", len(eksik_temel_kayitlar))
print("Kalan kayıt sayısı:", len(temiz_veri))

Playtest sonrası kayıt sayısı: 117716
Çıkarılan eksik alanlı kayıt: 1435
Kalan kayıt sayısı: 116281


## Güçlü oyun dışı adayların çıkarılması

Temel oyun türlerinden hiçbirini içermeyen ve yazılım, tasarım, video, ses,
eğitim veya geliştirme aracı niteliği taşıyan kayıtlar temizlenmiş öneri
havuzuna alınmayacaktır.

In [4]:
oyun_disi_turler = [
    "Utilities",
    "Design & Illustration",
    "Animation & Modeling",
    "Video Production",
    "Game Development",
    "Audio Production",
    "Software Training",
    "Photo Editing",
    "Web Publishing",
    "Accounting",
    "Movie",
    "Documentary",
    "Episodic",
    "Short",
    "Tutorial",
    "360 Video"
]

oyun_disi_deseni = "|".join(oyun_disi_turler)

temel_oyun_turleri = (
    "Indie|"
    "Casual|"
    "Action|"
    "Adventure|"
    "Simulation|"
    "Strategy|"
    "RPG|"
    "Sports|"
    "Racing|"
    "Massively Multiplayer"
)

oyun_disi_mi = temiz_veri["Genres"].str.contains(
    oyun_disi_deseni,
    case=False,
    na=False
)

temel_oyun_turu_var_mi = temiz_veri["Genres"].str.contains(
    temel_oyun_turleri,
    case=False,
    na=False
)

guclu_oyun_disi_mi = oyun_disi_mi & (
    temel_oyun_turu_var_mi == False
)

guclu_oyun_disi_kayitlar = temiz_veri[
    guclu_oyun_disi_mi
]

temiz_veri = temiz_veri[
    guclu_oyun_disi_mi == False
].copy()

print(
    "Çıkarılan güçlü oyun dışı aday:",
    len(guclu_oyun_disi_kayitlar)
)

print(
    "Kalan kayıt sayısı:",
    len(temiz_veri)
)

Çıkarılan güçlü oyun dışı aday: 576
Kalan kayıt sayısı: 115705


## Dedicated Server ve SDK kayıtlarının çıkarılması

Dedicated Server ve SDK kayıtları oynanabilir oyun olmadığı için temizlenmiş
öneri havuzuna alınmayacaktır. Editor ifadesi ise bazı gerçek oyunların
adında da bulunduğu için tek başına çıkarma ölçütü olarak kullanılmayacaktır.

In [5]:
sunucu_sdk_mi = temiz_veri["Name"].str.contains(
    "Dedicated Server|SDK",
    case=False,
    na=False
)

sunucu_sdk_kayitlar = temiz_veri[
    sunucu_sdk_mi
]

temiz_veri = temiz_veri[
    sunucu_sdk_mi == False
].copy()

print(
    "Çıkarılan Dedicated Server ve SDK kaydı:",
    len(sunucu_sdk_kayitlar)
)

print(
    "Kalan kayıt sayısı:",
    len(temiz_veri)
)

Çıkarılan Dedicated Server ve SDK kaydı: 0
Kalan kayıt sayısı: 115705


## Bilinmeyen dil bilgilerinin işaretlenmesi

Desteklenen dil listesi boş olan kayıtlar silinmeyecek, ancak dilleri
`Bilinmiyor` olarak işaretlenecektir. Belirli bir dil filtresi istendiğinde
bu kayıtlar doğrulanmış eşleşme olarak kullanılmayacaktır.

In [6]:
bos_dil_mi = temiz_veri[
    "Supported languages"
] == "[]"

print(
    "Bilinmeyen dil bilgisine sahip kayıt:",
    bos_dil_mi.sum()
)

temiz_veri.loc[
    bos_dil_mi,
    "Supported languages"
] = "Bilinmiyor"

print(
    "Bilinmeyen dil bilgisi işaretlendi."
)

Bilinmeyen dil bilgisine sahip kayıt: 97
Bilinmeyen dil bilgisi işaretlendi.


## Temizleme sürecinin özeti

Temizleme kuralları sırayla uygulanmış ve her adımdan sonra kalan kayıt sayısı
kontrol edilmiştir. Ham veri dosyası değiştirilmemiştir.

In [9]:
print("Başlangıç kayıt sayısı:", len(veri))
print("Çıkarılan Playtest:", len(playtest_kayitlari))
print("Çıkarılan eksik temel alanlı kayıt:", len(eksik_temel_kayitlar))
print("Çıkarılan güçlü oyun dışı aday:", len(guclu_oyun_disi_kayitlar))
print("Çıkarılan Dedicated Server ve SDK:", len(sunucu_sdk_kayitlar))
print("Son temiz kayıt sayısı:", len(temiz_veri))
print("Son sütun sayısı:", len(temiz_veri.columns))

print("\nTemel alanlardaki eksik değerler:")
print(temiz_veri[temel_alanlar].isna().sum())

print(
    "\nBilinmiyor olarak işaretlenen dil sayısı:",
    (temiz_veri["Supported languages"] == "Bilinmiyor").sum()
)

Başlangıç kayıt sayısı: 125855
Çıkarılan Playtest: 8139
Çıkarılan eksik temel alanlı kayıt: 1435
Çıkarılan güçlü oyun dışı aday: 576
Çıkarılan Dedicated Server ve SDK: 0
Son temiz kayıt sayısı: 115705
Son sütun sayısı: 40

Temel alanlardaki eksik değerler:
Name              0
About the game    0
Categories        0
Genres            0
dtype: int64

Bilinmiyor olarak işaretlenen dil sayısı: 97


In [10]:
from pathlib import Path

processed_klasor = Path("../data/processed")
processed_klasor.mkdir(
    parents=True,
    exist_ok=True
)

cikti_yolu = processed_klasor / "temiz_games.csv"

temiz_veri.to_csv(
    cikti_yolu,
    index=False
)

print("Temiz veri kaydedildi:", cikti_yolu)
print("Kaydedilen kayıt sayısı:", len(temiz_veri))

Temiz veri kaydedildi: ..\data\processed\temiz_games.csv
Kaydedilen kayıt sayısı: 115705


## Temiz verinin doğrulanması

Kaydedilen temiz veri dosyası yeniden okunarak kayıt sayısı, temel alanlardaki
eksiklikler, Playtest kayıtları ve bilinmeyen dil bilgileri kontrol edilmiştir.

In [11]:
temiz_veri_kontrol = pd.read_csv(
    cikti_yolu,
    usecols=[
        "Name",
        "About the game",
        "Categories",
        "Genres",
        "Supported languages"
    ]
)

print("Yeniden okunan kayıt sayısı:", len(temiz_veri_kontrol))

print(
    "Temel alanlardaki toplam eksik değer:",
    temiz_veri_kontrol[
        temel_alanlar
    ].isna().sum().sum()
)

print(
    "Playtest içeren kayıt:",
    temiz_veri_kontrol["Name"].str.contains(
        "Playtest",
        case=False,
        na=False
    ).sum()
)

print(
    "Boş dil listesi kalan kayıt:",
    (
        temiz_veri_kontrol["Supported languages"] == "[]"
    ).sum()
)

print(
    "Bilinmiyor dil bilgisi:",
    (
        temiz_veri_kontrol["Supported languages"] == "Bilinmiyor"
    ).sum()
)

Yeniden okunan kayıt sayısı: 115705
Temel alanlardaki toplam eksik değer: 0
Playtest içeren kayıt: 0
Boş dil listesi kalan kayıt: 0
Bilinmiyor dil bilgisi: 97


Temizleme işlemi sonucunda 125.855 kayıttan 115.705 kayıt öneri sisteminde
kullanılmak üzere bırakılmıştır. Playtest kayıtları, temel alanları eksik
kayıtlar ve güçlü oyun dışı adaylar temizleme dışında tutulmuştur.

Temiz veri dosyası yeniden okunduğunda temel alanlarda eksik değer, Playtest
kaydı veya boş dil listesi kalmadığı görülmüştür. Dil bilgisi bulunmayan 97 kayıt
`Bilinmiyor` olarak işaretlenmiştir.

Temiz veri `data/processed/temiz_games.csv` konumuna kaydedilmiştir.